# 06 - X-Ray Model Training

This notebook trains a CNN classifier for chest X-ray classification.

**Goals:**
1. Load and preprocess the X-ray dataset
2. Train a DenseNet121 model with transfer learning
3. Evaluate model performance
4. Visualize results

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

import torch
import numpy as np
import matplotlib.pyplot as plt
import yaml
import json

print(f"PyTorch version: {torch.__version__}")
device = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

## 1. Configuration

In [ ]:
# Load configuration
config_path = project_root / "config_xray.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

print("Configuration:")
print(f"  Backbone: {config['model']['backbone']}")
print(f"  Classes: {config['dataset']['classes']}")
print(f"  Epochs: {config['training']['epochs']}")
print(f"  Batch size: {config['training']['batch_size']}")
print(f"  Learning rate: {config['training']['learning_rate']}")

## 2. Check Dataset

In [ ]:
# Find dataset
data_dir = project_root / "data" / "xray"
expected_path = data_dir / "COVID-19_Radiography_Dataset"

if expected_path.exists():
    dataset_path = expected_path
    print(f"Dataset found at: {dataset_path}")
    
    # List classes
    classes = [d.name for d in dataset_path.iterdir() if d.is_dir()]
    print(f"Classes found: {classes}")
else:
    print(f"Dataset NOT found at: {expected_path}")
    print("\nPlease download the dataset first:")
    print("https://www.kaggle.com/datasets/tawsifurrahman/covid19-radiography-database")
    dataset_path = None

## 3. Load Data

In [ ]:
if dataset_path:
    from src.image_dataset import create_data_loaders
    
    train_loader, val_loader, test_loader, class_names = create_data_loaders(
        root_dir=str(dataset_path),
        class_names=config['dataset']['classes'],
        train_ratio=config['dataset']['train_ratio'],
        val_ratio=config['dataset']['val_ratio'],
        test_ratio=config['dataset']['test_ratio'],
        batch_size=config['training']['batch_size'],
        image_size=config['preprocessing']['image_size'],
        seed=config['training']['seed'],
    )
    
    print(f"\nClass names: {class_names}")
else:
    print("Cannot load data - dataset not found.")

In [ ]:
# Visualize a batch
if dataset_path:
    images, labels = next(iter(train_loader))
    
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    
    for i, ax in enumerate(axes.flat):
        if i < len(images):
            # Denormalize for visualization
            img = images[i].permute(1, 2, 0).numpy()
            img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
            img = np.clip(img, 0, 1)
            
            ax.imshow(img)
            ax.set_title(class_names[labels[i]])
            ax.axis('off')
    
    plt.suptitle('Sample Training Images', fontsize=14)
    plt.tight_layout()
    plt.show()

## 4. Create Model

In [ ]:
if dataset_path:
    from src.image_model import create_model
    
    model, device = create_model(
        num_classes=len(class_names),
        backbone=config['model']['backbone'],
        pretrained=config['model']['pretrained'],
        dropout=config['model']['dropout'],
    )

## 5. Train Model

For full training, use the command line:
```bash
python src/image_train.py --config config_xray.yaml
```

Or train here with reduced epochs for quick testing:

In [ ]:
# Quick training (reduced epochs for notebook)
QUICK_TRAIN = True  # Set to False for full training

if dataset_path and QUICK_TRAIN:
    import torch.nn as nn
    import torch.optim as optim
    from tqdm.notebook import tqdm
    from src.image_dataset import compute_class_weights
    
    # Setup
    train_labels = [label for _, label in train_loader.dataset]
    class_weights = compute_class_weights(train_labels, len(class_names)).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.Adam(model.parameters(), lr=config['training']['learning_rate'])
    
    # Quick training loop (3 epochs)
    num_epochs = 3
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    
    print(f"Quick training for {num_epochs} epochs...")
    
    for epoch in range(1, num_epochs + 1):
        # Train
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}")
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()
            
            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{train_correct/train_total:.3f}'})
        
        # Validate
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()
        
        # Record
        history['train_loss'].append(train_loss / len(train_loader))
        history['train_acc'].append(train_correct / train_total)
        history['val_loss'].append(val_loss / len(val_loader))
        history['val_acc'].append(val_correct / val_total)
        
        print(f"Epoch {epoch}: Train Acc={train_correct/train_total:.3f}, Val Acc={val_correct/val_total:.3f}")

In [ ]:
# Plot training history
if dataset_path and QUICK_TRAIN and len(history['train_loss']) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Loss
    axes[0].plot(history['train_loss'], label='Train')
    axes[0].plot(history['val_loss'], label='Validation')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training Loss')
    axes[0].legend()
    
    # Accuracy
    axes[1].plot(history['train_acc'], label='Train')
    axes[1].plot(history['val_acc'], label='Validation')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Training Accuracy')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()

## 6. Evaluate Model

In [ ]:
if dataset_path:
    from sklearn.metrics import classification_report, confusion_matrix
    import seaborn as sns
    
    # Evaluate on test set
    model.eval()
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Testing"):
            images = images.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
    
    # Classification report
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=class_names))
    
    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.tight_layout()
    plt.show()

## 7. Save Model

In [ ]:
if dataset_path:
    from src.image_model import save_model
    from datetime import datetime
    
    # Save model
    output_dir = project_root / "outputs" / "xray_models"
    output_dir.mkdir(parents=True, exist_ok=True)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    model_path = output_dir / f"model_{timestamp}.pt"
    
    save_model(model, str(model_path), class_names, config)
    print(f"\nModel saved to: {model_path}")

## 8. Test Inference

In [ ]:
if dataset_path and 'model_path' in dir():
    from src.image_predict import XRayPredictor
    
    # Load predictor
    predictor = XRayPredictor(str(model_path))
    
    # Test on a sample image
    # Get a test image
    test_image_path = None
    for class_dir in dataset_path.iterdir():
        if class_dir.is_dir():
            images_dir = class_dir / "images" if (class_dir / "images").exists() else class_dir
            for img in images_dir.glob("*.png"):
                test_image_path = str(img)
                break
        if test_image_path:
            break
    
    if test_image_path:
        result = predictor.predict(test_image_path)
        
        print(f"\nTest Image: {test_image_path}")
        print(f"Prediction: {result.predicted_class}")
        print(f"Confidence: {result.confidence:.1%}")
        print(f"\nAll probabilities:")
        for cls, prob in sorted(result.all_probabilities.items(), key=lambda x: -x[1]):
            print(f"  {cls}: {prob:.1%}")

## Summary

**For full training**, use the command line:
```bash
python src/image_train.py --config config_xray.yaml
```

**For inference**:
```bash
python src/image_predict.py --model-path outputs/xray_models/best_model.pt --image path/to/xray.png
```